In [1]:
import os
import yaml
import tqdm
from dotenv import load_dotenv
from src.prompt_builder import PromptBuilder
from src.clients.gemini_client import GeminiClient
from src.clients.ollama_cloud_client import OllamaCloudClient
from src.clients.ollama_local_client import OllamaLocalClient

In [2]:
with open("data/questions.yaml", "r") as file:
    questions = yaml.safe_load(file)

In [ ]:
# Uncomment the type of client you want to use
load_dotenv()
# client = GeminiClient(api_key = os.getenv("GEMINI_API_KEY"))
# client = OllamaCloudClient(api_key = os.getenv("OLLAMA_API_KEY"))
client = OllamaLocalClient(model="qwen3:4b")

In [4]:
builder = PromptBuilder()

outputs = [client.model]
for question_id in questions:
    output = {}

    question = questions[question_id]["question"]
    output["question"] = question

    prompt1 = builder.build_prompt_with_memory(question)
    output["answer_neutral"] = client.get_response(prompt1)

    prompt2 = builder.build_prompt_with_memory(question, builder.positive_memory)
    output["answer_positive"] = client.get_response(prompt2)

    prompt3 = builder.build_prompt_with_memory(question, builder.negative_memory)
    output["answer_negative"] = client.get_response(prompt3)

    outputs.append(output)

with open("data/answers.yaml", "w") as file:
    yaml.dump(outputs, file)